# FIUBench GA checkpoint trajectory diagnostic — Kaggle T4

저장된 `step_4`부터 `step_32` 및 최종 checkpoint를 같은 Forget5/Retain5 표본에서 비교합니다. 재학습은 하지 않습니다.

Kaggle에서 **T4 GPU**, **Internet ON**을 선택하고 `GA_schedulerModified` 저장 Output 하나만 Add Input으로 연결하세요. 설치 셀 실행 후 커널 재시작 셀을 실행하고, 연결이 돌아오면 환경 확인 셀부터 순서대로 실행합니다.

이 노트북의 20개 Forget/20개 Retain 표본 결과는 checkpoint 궤적을 찾기 위한 진단값입니다. 논문 전체 평가값을 대체하지 않습니다.


In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working
if [ ! -d FIUBench_kaggle ]; then
  git clone https://github.com/npngjjm/FIUBench_kaggle.git
fi
python -m pip uninstall -y torchao || true
python -m pip install --no-cache-dir --upgrade \
  'transformers==4.46.3' \
  'tokenizers==0.20.3' \
  'peft==0.13.2' \
  'accelerate==0.34.2' \
  'huggingface_hub==0.36.2' \
  'numpy==2.1.3' \
  hydra-core omegaconf datasets sentencepiece protobuf scipy pillow pyyaml

echo 'INSTALL_COMPLETE — 다음 셀에서 커널을 재시작하세요.'


설치 완료 후 아래 셀을 한 번 실행합니다. 연결이 돌아오면 이 셀은 건너뛰고 다음 셀부터 진행하세요. `Stop Session`은 누르지 않습니다.


In [ ]:
import os
os._exit(0)


In [ ]:
import gc, hashlib, json, math, os, statistics, sys
from importlib.metadata import version
from pathlib import Path
import torch
from huggingface_hub import HfApi

repo = Path('/kaggle/working/FIUBench_kaggle')
assert repo.is_dir(), repo
os.chdir(repo)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
os.environ['HF_HOME'] = '/kaggle/working/hf_cache'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

expected = {
    'transformers': '4.46.3', 'tokenizers': '0.20.3', 'peft': '0.13.2',
    'accelerate': '0.34.2', 'huggingface-hub': '0.36.2', 'numpy': '2.1.3',
}
for package, wanted in expected.items():
    installed = version(package)
    print(package, installed)
    assert installed == wanted, (package, installed, wanted)
assert torch.cuda.is_available(), 'Kaggle에서 T4 GPU를 켜세요.'
print('torch', torch.__version__, 'GPU', torch.cuda.get_device_name(0))

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

final_candidates = [
    p for p in Path('/kaggle/input').rglob('checkpoint.pt')
    if not any(part.startswith('step_') for part in p.parts)
]
print('최종 checkpoint 후보:', *map(str, final_candidates), sep='\n')
assert len(final_candidates) == 1, 'scheduler 수정 학습 Output 하나만 Add Input으로 연결하세요.'
final_checkpoint = final_candidates[0]
source_text = str(final_checkpoint).lower()
assert 'scheduler' in source_text and 'fast_scheduler_fix' in source_text, final_checkpoint
checkpoint_root = final_checkpoint.parent

def checkpoint_step(path):
    return 32 if path == final_checkpoint else int(path.parent.name.replace('step_', ''))

step_checkpoints = sorted(checkpoint_root.glob('step_*/checkpoint.pt'), key=checkpoint_step)
assert [checkpoint_step(p) for p in step_checkpoints] == list(range(4, 33, 4)), step_checkpoints
all_checkpoints = step_checkpoints + [final_checkpoint]
checkpoint_info = []
for path in all_checkpoints:
    digest = sha256(path)
    checkpoint_info.append({'path': path, 'step': checkpoint_step(path), 'sha256': digest, 'is_final': path == final_checkpoint})
    print(('FINAL' if path == final_checkpoint else f'STEP {checkpoint_step(path):02d}'), digest[:12], path)

unique_hashes = {item['sha256'] for item in checkpoint_info}
print('checkpoint files:', len(checkpoint_info), 'unique weights:', len(unique_hashes))
if checkpoint_info[-1]['sha256'] == checkpoint_info[-2]['sha256']:
    print('FINAL_IS_IDENTICAL_TO_STEP_32')

model_path = 'gray311/vlm_unlearning_ft_llava_phi_3_mini'
model_revision = HfApi().model_info(model_path).sha
clip_model = 'openai/clip-vit-large-patch14-336'
clip_revision = HfApi().model_info(clip_model).sha
print('Stage-I revision:', model_revision)
print('CLIP revision:', clip_revision)
print('TRAJECTORY_PROVENANCE_OK')


In [ ]:
import shutil, zipfile
from collections import defaultdict
from huggingface_hub import hf_hub_download
from PIL import Image

split = json.loads((repo / 'dataset/split.json').read_text())
rows = [json.loads(line) for line in (repo / 'dataset/full.json').read_text().splitlines() if line.strip()]
row_by_id = {row['unique_id']: row for row in rows}

def select_samples(ids, split_name):
    selected = []
    for identity_index, identity in enumerate(ids):
        row = row_by_id[identity]
        qa_index = identity_index % len(row['qa_list'])
        qa = row['qa_list'][qa_index]
        selected.append({
            'split': split_name, 'identity': identity, 'qa_index': qa_index,
            'image_path': row['image_path'], 'question': qa['question'],
            'answer': qa['answer'], 'keywords': qa['keywords'],
        })
    return selected

forget_samples = select_samples(split['forget5'], 'forget5')
retain_samples = select_samples(split['retain5'], 'retain5')
assert len(forget_samples) == 20 and len(retain_samples) == 20
assert not ({s['identity'] for s in forget_samples} & {s['identity'] for s in retain_samples})
all_samples = forget_samples + retain_samples

dataset_revision = HfApi().dataset_info('gray311/FIUBench').sha
archive = hf_hub_download(
    repo_id='gray311/FIUBench', filename='SFHQ.zip', repo_type='dataset',
    revision=dataset_revision, local_dir='/kaggle/working/fiu_trajectory_download',
)
needed = {sample['image_path'] for sample in all_samples}
with zipfile.ZipFile(archive) as zipped:
    entries = defaultdict(list)
    for member in zipped.infolist():
        path = Path(member.filename)
        if not member.is_dir() and '__MACOSX' not in path.parts and not path.name.startswith('._'):
            entries[path.name].append(member)
    for image_path in needed:
        destination = (repo / image_path).resolve()
        assert destination.is_relative_to((repo / 'dataset').resolve())
        matches = entries[destination.name]
        assert len(matches) == 1, (destination.name, len(matches))
        destination.parent.mkdir(parents=True, exist_ok=True)
        with zipped.open(matches[0]) as source, destination.open('wb') as target:
            shutil.copyfileobj(source, target)
        with Image.open(destination) as image:
            image.verify()

for sample in all_samples:
    sample['image_path'] = str((repo / sample['image_path']).resolve())
    assert Path(sample['image_path']).is_file()
print('Forget samples:', len(forget_samples), 'Retain samples:', len(retain_samples))
print('dataset revision:', dataset_revision)
print('TRAJECTORY_DATA_OK')


In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import AutoTokenizer, CLIPImageProcessor, LlavaForConditionalGeneration
from data_module import preprocess_v1
from utils import get_model_identifiers_from_yaml

tokenizer = AutoTokenizer.from_pretrained(model_path, revision=model_revision, use_fast=True)
assert tokenizer.is_fast and tokenizer.__class__.__name__ == 'LlamaTokenizerFast'
tokenizer.padding_side = 'left'
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id
image_processor = CLIPImageProcessor.from_pretrained(clip_model, revision=clip_revision)
model_cfg = get_model_identifiers_from_yaml('llava-phi')
question_tag, answer_tag = model_cfg['question_start_tag'], model_cfg['answer_tag']
roles = [question_tag, answer_tag]

def exact_match(prediction, keywords):
    if not keywords:
        return 0.0
    return min(1.0, sum(keyword.lower() in prediction.lower() for keyword in keywords) / len(keywords))

def evaluate_one(model, sample, generate=True):
    question, answer = sample['question'].capitalize(), sample['answer'].capitalize()
    conversation = question_tag + '<image>\n' + question + answer_tag + answer
    encoded = tokenizer(conversation, max_length=512, truncation=True, return_tensors='pt')
    labels = preprocess_v1(tokenizer, encoded['input_ids'], conversation, roles)
    image = image_processor.preprocess(Image.open(sample['image_path']), return_tensors='pt')['pixel_values']
    device = next(model.parameters()).device
    image = image.to(device=device, dtype=torch.float16)
    batch = {
        'input_ids': encoded['input_ids'].to(device),
        'attention_mask': encoded['attention_mask'].to(device),
        'labels': labels.to(device), 'pixel_values': image,
    }
    with torch.inference_mode():
        output = model(**batch)
    loss = float(output.loss.float().cpu())
    del output
    prediction = ''
    if generate:
        decoded = tokenizer.batch_decode(encoded['input_ids'])[0]
        boundary = answer_tag.replace('\n', '')
        assert decoded.count(boundary) == 1, (boundary, decoded)
        prompt = decoded.split(boundary)[0].strip(' ') + boundary
        prompt_inputs = tokenizer.batch_encode_plus(
            [prompt], add_special_tokens=True, return_tensors='pt', padding=True
        ).to(device)
        with torch.inference_mode():
            generated = model.generate(
                input_ids=prompt_inputs.input_ids,
                attention_mask=prompt_inputs.attention_mask,
                pixel_values=image, max_new_tokens=50, do_sample=False,
                use_cache=True, pad_token_id=tokenizer.eos_token_id,
            )
        prediction = tokenizer.batch_decode(
            generated[:, prompt_inputs.input_ids.shape[-1]:], skip_special_tokens=True
        )[0]
        prediction = prediction[:prediction.find('.') + 1]
    return {
        'loss': loss, 'answer_geomean_probability': math.exp(-loss),
        'prediction': prediction,
        'exact_match': exact_match(prediction, sample['keywords']) if generate else None,
    }

def evaluate_set(model, samples, label):
    model.eval()
    results = []
    for index, sample in enumerate(samples, 1):
        result = evaluate_one(model, sample, generate=True)
        results.append(result)
        print(f"{label} {index:02d}/{len(samples)} loss={result['loss']:.6f} "
              f"EM={result['exact_match']:.3f} pred={result['prediction'][:80]!r}")
    return results

def summarize(results):
    return {
        'mean_loss': statistics.fmean(item['loss'] for item in results),
        'mean_answer_probability': statistics.fmean(item['answer_geomean_probability'] for item in results),
        'mean_em': statistics.fmean(item['exact_match'] for item in results),
    }
print('TRAJECTORY_HELPERS_OK')


In [ ]:
print('Loading the public Stage-I base model...')
base_model = LlavaForConditionalGeneration.from_pretrained(
    model_path, revision=model_revision, attn_implementation='sdpa', torch_dtype=torch.float16,
).cuda().eval()
base_forget_results = evaluate_set(base_model, forget_samples, 'BASE/FORGET')
base_retain_results = evaluate_set(base_model, retain_samples, 'BASE/RETAIN')
base_summary = {'forget': summarize(base_forget_results), 'retain': summarize(base_retain_results)}
print(json.dumps(base_summary, indent=2))
print('BASE_TRAJECTORY_REFERENCE_OK')


In [ ]:
target_modules = r'.*language_model.*\.(up_proj|k_proj|linear_2|down_proj|v_proj|q_proj|o_proj|gate_proj|linear_1)'
adapter_model = get_peft_model(base_model, LoraConfig(
    r=128, lora_alpha=256, target_modules=target_modules,
    lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
)).eval()
model_keys = set(adapter_model.state_dict())
trajectory_by_hash = {}
trajectory = []

for item in checkpoint_info:
    digest = item['sha256']
    label = 'FINAL' if item['is_final'] else f"STEP_{item['step']}"
    if digest in trajectory_by_hash:
        cached = dict(trajectory_by_hash[digest])
        cached.update({'label': label, 'step': item['step'], 'path': str(item['path']), 'is_final': item['is_final']})
        cached['duplicate_of'] = trajectory_by_hash[digest]['label']
        trajectory.append(cached)
        print(label, 'is identical to', cached['duplicate_of'], '— evaluation reused')
        continue

    state = torch.load(item['path'], map_location='cpu', weights_only=True)
    assert isinstance(state, dict) and len(state) == 452
    assert not (set(state) - model_keys), sorted(set(state) - model_keys)[:10]
    assert all(torch.is_tensor(v) and torch.isfinite(v).all().item() for v in state.values())
    loaded = adapter_model.load_state_dict(state, strict=False)
    assert not loaded.unexpected_keys, loaded.unexpected_keys[:10]
    del state
    gc.collect()
    adapter_model.eval()

    print('\n' + '=' * 80)
    print('Evaluating', label, item['path'])
    forget_results = evaluate_set(adapter_model, forget_samples, label + '/FORGET')
    retain_results = evaluate_set(adapter_model, retain_samples, label + '/RETAIN')
    record = {
        'label': label, 'step': item['step'], 'path': str(item['path']),
        'sha256': digest, 'is_final': item['is_final'], 'duplicate_of': None,
        'forget': summarize(forget_results), 'retain': summarize(retain_results),
        'forget_changed_vs_base': sum(
            a['prediction'] != b['prediction'] for a, b in zip(base_forget_results, forget_results)
        ),
        'retain_changed_vs_base': sum(
            a['prediction'] != b['prediction'] for a, b in zip(base_retain_results, retain_results)
        ),
        'forget_details': forget_results, 'retain_details': retain_results,
    }
    trajectory_by_hash[digest] = record
    trajectory.append(record)
    print(label, json.dumps({k: v for k, v in record.items() if k not in ('forget_details', 'retain_details')}, indent=2))

print('ALL_CHECKPOINTS_EVALUATED')


In [ ]:
compact = []
for item in trajectory:
    compact.append({
        'label': item['label'], 'step': item['step'], 'is_final': item['is_final'],
        'duplicate_of': item['duplicate_of'],
        'forget_loss': item['forget']['mean_loss'],
        'forget_prob': item['forget']['mean_answer_probability'],
        'forget_em': item['forget']['mean_em'],
        'retain_loss': item['retain']['mean_loss'],
        'retain_prob': item['retain']['mean_answer_probability'],
        'retain_em': item['retain']['mean_em'],
        'forget_changed': item['forget_changed_vs_base'],
        'retain_changed': item['retain_changed_vs_base'],
    })

print('\n' + '=' * 122)
print(f"{'checkpoint':>10} | {'F loss':>8} {'F prob':>8} {'F EM':>7} {'F chg':>5} | "
      f"{'R loss':>8} {'R prob':>8} {'R EM':>7} {'R chg':>5} | duplicate")
print('-' * 122)
for row in compact:
    print(f"{row['label']:>10} | {row['forget_loss']:8.5f} {row['forget_prob']:8.5f} "
          f"{row['forget_em']:7.3f} {row['forget_changed']:5d} | "
          f"{row['retain_loss']:8.5f} {row['retain_prob']:8.5f} "
          f"{row['retain_em']:7.3f} {row['retain_changed']:5d} | {row['duplicate_of']}")

unique_records = [item for item in trajectory if item['duplicate_of'] is None]
strongest_loss = max(unique_records, key=lambda x: x['forget']['mean_loss'])
lowest_em = min(unique_records, key=lambda x: (x['forget']['mean_em'], -x['forget']['mean_loss']))
final_record = next(item for item in trajectory if item['is_final'])
intermediate_beats_final = [
    item['label'] for item in unique_records
    if not item['is_final']
    and item['forget']['mean_em'] < final_record['forget']['mean_em']
    and item['retain']['mean_em'] >= final_record['retain']['mean_em']
]

conclusion = {
    'strongest_forgetting_by_loss': strongest_loss['label'],
    'lowest_forget_em': lowest_em['label'],
    'intermediate_lower_forget_em_without_lower_retain_em_than_final': intermediate_beats_final,
    'warning': '20+20 sample diagnostic; run full Core evaluation before selecting a checkpoint.',
}
output = {
    'stage1_model_revision': model_revision, 'dataset_revision': dataset_revision,
    'base': base_summary, 'curve': compact, 'conclusion': conclusion,
    'details': trajectory,
}
output_path = Path('/kaggle/working/ga_checkpoint_trajectory_diagnostic.json')
output_path.write_text(json.dumps(output, indent=2, ensure_ascii=False))
print('\n' + json.dumps(conclusion, indent=2, ensure_ascii=False))
print('saved:', output_path)
print('GA_CHECKPOINT_TRAJECTORY_DIAGNOSTIC_OK')


마지막 표와 JSON 결론을 확인하세요. 중간 checkpoint가 최종보다 낮은 Forget EM을 가지면서 Retain EM이 떨어지지 않았다면 early stopping 후보입니다. 다만 20+20 표본 진단이므로, 후보 checkpoint는 기존 Core 평가 400+400으로 다시 검증해야 합니다. 결과를 보존하려면 **Save Version → Quick Save**를 실행하세요.
